# LangGraph G15 — The full system
Everything you built, in one graph. This is the shape production agent systems converge on:
identity and routing in front, an agent only where judgement is needed, approval on writes,
persistence underneath, traces beside.

```text
START -> load_profile -> manage_context -> triage --+-> faq desk (retrieval, grounded)           --+
                                                    +-> records desk (role filter, guard,          |
                                                    |     approval, retries, local + MCP tools)   +-> END
                                                    +-> eligibility desk (parallel checks)       --+
                                                    +-> smalltalk                                --+
   checkpointer (threads) + store (people) underneath; stream() and state history beside
```

### Step 1 — Assemble CampusAI

Nothing here is new: every node comes from an earlier section. The records desk is the safe
agent from G8 with the retry policy from G9 and the MCP tools from G11, so the whole system runs
with `ainvoke`.

In [ ]:
class CampusState(TypedDict, total=False):                 # ours: the outer state = every earlier state key
    messages: Annotated[list, add_messages]
    summary: str
    profile: list
    category: str
    priority: str
    verdict: str

FINAL_TOOLS = KNOWLEDGE_TOOLS + WRITE_TOOLS + mcp_tools + [remember_about_me]
ROLE_TOOLS = {"student": {t.name for t in KNOWLEDGE_TOOLS + mcp_tools + [remember_about_me]}, "staff": {t.name for t in FINAL_TOOLS}}

def tools_for(role):                                       # ours: updated for the full tool list
    return [t for t in FINAL_TOOLS if t.name in ROLE_TOOLS.get(role, set())]

def final_agent(state: CampusState, runtime: Runtime[Context]):   # ours: G8 agent + G5 summary + G6 profile
    facts = "; ".join(state.get("profile") or []) or "none yet"
    persona = CAMPUS_PERSONA + f" Known facts about this user: {facts}\nFollow the user's stated preferences. Look up the student and the course before registering. Report rejections honestly."
    if state.get("summary"):
        persona += f" Summary of earlier conversation: {state['summary']}"
    reply = model.bind_tools(tools_for(runtime.context.role)).invoke([SystemMessage(persona)] + state["messages"])   # LangChain
    return {"messages": [reply]}

rg = StateGraph(CampusState, context_schema=Context)      # the records desk: G8 shape, G9 retries, G11 tools
rg.add_node("agent", final_agent)
rg.add_node("tools", ToolNode(FINAL_TOOLS, handle_tool_errors=True), retry_policy=RetryPolicy(max_attempts=3, initial_interval=0.1, retry_on=TimeoutError))   # LangGraph
rg.add_node("guard", guard)
rg.add_node("approval", approval)
rg.add_edge(START, "agent")
rg.add_conditional_edges("agent", route_after_agent, {"tools": "tools", "guard": "guard", END: END})
rg.add_conditional_edges("guard", after_check, {"agent": "agent", "next": "approval"})
rg.add_conditional_edges("approval", after_check, {"agent": "agent", "next": "tools"})
rg.add_edge("tools", "agent")
records_desk = rg.compile()                                # LangGraph: subgraphs inherit the parent's checkpointer and store

class FullRoute(BaseModel):                                # ours
    """Which desk handles the message."""
    category: Literal["faq", "records", "eligibility", "smalltalk"] = Field(description="eligibility when the user asks whether a student may take a course; records for facts, registrations, emails or the library; faq for rules and general information; smalltalk otherwise.")

def full_triage(state: CampusState):
    text = text_of(state["messages"][-1])
    if re.search(r"eligible|can .* take|allowed to register", text.lower()) and re.search(r"S\d{3}", text) and re.search(r"[A-Z]{2}\d{3}", text):
        return {"category": "eligibility", "priority": "medium"}   # ours: a deterministic rule first; the model decides the rest
    ticket = structured(Ticket).invoke([HumanMessage(text)])       # LangChain
    return {"category": ticket.category, "priority": ticket.priority}

def eligibility_desk(state: CampusState):                  # ours: run the G10 graph and report as a message
    text = text_of(state["messages"][-1])
    result = eligibility.invoke({"student_id": re.search(r"S\d{3}", text).group(0), "course_code": re.search(r"[A-Z]{2}\d{3}", text).group(0), "findings": []})   # LangGraph
    return {"messages": [AIMessage(content=result["verdict"])], "verdict": result["verdict"]}

g = StateGraph(CampusState, context_schema=Context)
g.add_node("load_profile", load_profile)                   # G6
g.add_node("manage_context", manage_context)               # G5
g.add_node("triage", full_triage)                          # G4
g.add_node("faq", faq_rag)                                 # G7
g.add_node("records", records_desk)                        # G3, G8, G9, G11 (subgraph node)
g.add_node("eligibility", eligibility_desk)                # G10
g.add_node("smalltalk", smalltalk)                         # G2
g.add_edge(START, "load_profile")
g.add_edge("load_profile", "manage_context")
g.add_edge("manage_context", "triage")
g.add_conditional_edges("triage", lambda s: s["category"], {"faq": "faq", "records": "records", "eligibility": "eligibility", "smalltalk": "smalltalk"})
for node in ("faq", "records", "eligibility", "smalltalk"):
    g.add_edge(node, END)
campusai = g.compile(checkpointer=InMemorySaver(), store=store)   # LangGraph: the whole system, persistent, with long-term memory
print(campusai.get_graph().draw_mermaid())

### Step 2 — One staff conversation through every desk

The same thread carries five turns: a memory is saved, a rule is retrieved, an eligibility check
runs in parallel, the library server answers, and a registration pauses for approval. Then the
audit trail lists every node that ran.

In [ ]:
thread = {"configurable": {"thread_id": "final-1"}}
staff = Context(user_id="staff-7", role="staff")
turns = [
    "Please remember that I prefer short bullet-point answers.",
    "What is the late registration rule?",
    "Is S002 eligible to take EE150?",
    "When is the main library open?",
    "Register student S002 for course EE150.",
]
for q in turns:
    out = await campusai.ainvoke({"messages": [HumanMessage(q)]}, thread, context=staff)   # LangGraph: async because MCP tools are async
    if "__interrupt__" in out:                              # LangGraph: the records desk paused for approval
        print(f"[records] PAUSED for approval: {out['__interrupt__'][0].value['actions']}")
        out = await campusai.ainvoke(Command(resume=True), thread, context=staff)
    this_turn = out["messages"][max(i for i, m in enumerate(out["messages"]) if isinstance(m, HumanMessage)):]   # ours: only this turn's messages
    used = [c["name"] for m in this_turn if isinstance(m, AIMessage) for c in m.tool_calls]
    print(f"[{out['category']}] {q}\n   tools: {used or '-'}\n   -> {text_of(out['messages'][-1])[:120]}")

print("\nAUDIT TRAIL (nodes that ran on this thread, oldest first):")
print("  ", [snap.next[0] for snap in reversed(list(campusai.get_state_history(thread))) if snap.next])   # LangGraph
print("registrations:", REGISTRATIONS)
print("profile in store:", store.get(("profiles", "staff-7"), "facts").value)   # LangGraph store

### Step 3 — What you built, and where it goes next

```text
                         USER
                           |
                  +-----------------+
                  | API / frontend  |   stream() for progress                       (G13)
                  +--------+--------+
                           |
                  +-----------------+
                  | identity, roles |   runtime context: user_id, role              (G6, G8)
                  +--------+--------+
                           |
        load_profile -> manage_context -> triage     memory, bounded context, routing (G4, G5, G6)
             +-------------+-------------+-----------+
             |             |             |           |
        faq desk     records agent   eligibility  smalltalk
       (retrieval,   (loop + guard    (parallel)
        grounded)    + approval +       (G10)
          (G7)       retries + MCP)
                    (G3, G8, G9, G11)
             +-----------------------------+
             | checkpointer + store        |   durable execution, audit trail        (G5, G6)
             +-----------------------------+
             | traces, evaluation, limits  |   observability, bounded autonomy       (G9, G13)
             +-----------------------------+
```

- **Persistence:** swap `InMemorySaver` and `InMemoryStore` for the Postgres implementations; the graph code does not change.
- **Deployment:** a compiled graph is a Python object; serve it behind FastAPI, or with LangGraph
  Server / LangGraph Platform, which add threads, streaming endpoints, background runs and cron jobs.
- **Specialists:** the supervisor or the handoff pair of G12 slots in as one more desk when a domain needs its own tools and owner.
- **Schedules:** the watcher of G14 is deployed as a cron job against the same checkpointer, and its queued approvals appear in the staff review screen.
- **Observability:** set `LANGSMITH_TRACING=true` and `LANGSMITH_API_KEY` to trace every node, tool
  call and token without changing code; keep the evaluation set of G13 in version control.
- **The question to ask first:** does this need an agent? If the steps are known, a workflow (G1, G4)
  is cheaper, faster and safer. Use the loop (G3) where judgement genuinely helps, and gate its writes (G8).

### Recap

- **Problem seen:** the layers lived in separate graphs.
- **Layer added:** one graph with profile loading, context management, triage, four desks, persistence, a store, MCP tools, approval and retries; the scheduled watcher of G14 runs beside it on its own threads.
- **Evidence:** a five-turn staff conversation crossed every desk, paused once for approval, and left a complete audit trail and a stored profile.